# MagiCanvas: three selected adapter experiments
Select your available T4 GPU. The unchanged full-length recipe runs sequentially, using FP16 on T4 and a per-model memory/numerical preflight. GPU fit and training quality are not yet verified. Upload the updated layout_training_bundle.zip and run in order. No public upload or automatic production promotion.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory >= 14 * 2**30, "Use a T4-class or larger CUDA GPU with at least 14 GiB visible VRAM."
print(gpu.name, round(gpu.total_memory / 2**30, 2), "GiB", torch.__version__)
print("Precision:", "BF16" if torch.cuda.is_bf16_supported(including_emulation=False) else "FP16", "— actual fit is checked per model before training.")
from pathlib import Path
Path("/content/torch-constraint.txt").write_text("torch==" + torch.__version__.split("+")[0] + "\n")

from google.colab import files
uploaded = files.upload()  # Select layout_training_bundle.zip
assert set(uploaded) == {"layout_training_bundle.zip"}

from pathlib import Path
from zipfile import ZipFile
bundle = Path("/content/layout_training")
bundle.mkdir(exist_ok=True)
with ZipFile("layout_training_bundle.zip") as z:
    assert all(not Path(n).is_absolute() and ".." not in Path(n).parts for n in z.namelist())
    z.extractall(bundle)


In [ ]:
%pip install torchvision --index-url https://download.pytorch.org/whl/cpu -c /content/torch-constraint.txt
%pip install -r /content/layout_training/requirements.txt -c /content/torch-constraint.txt


Choose **Runtime → Restart session**, confirm, then continue below. Uploaded/extracted files and installed packages remain. Do not choose **Disconnect and delete runtime** or rerun upload/install. If an older install warned about cuDF/PyArrow but finished successfully, continue: training does not use cuDF.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, psutil

drive.mount("/content/drive")
output = Path("/content/drive/MyDrive/MagiCanvas/audited_v2_candidates")
output.mkdir(parents=True, exist_ok=True)
script = "/content/layout_training/train.py"
# On a deliberate rerun, stop only an orphan from this exact training bundle.
# Saved checkpoints survive; progress since the last checkpoint may be lost.
old = []
for p in psutil.process_iter(["cmdline"]):
    command = p.info["cmdline"] or []
    if script in command and "--bundle" in command and command[command.index("--bundle") + 1] == "/content/layout_training":
        try:
            p.terminate()
            old.append(p)
        except psutil.NoSuchProcess:
            pass
_, alive = psutil.wait_procs(old, timeout=10)
for p in alive:
    p.kill()
psutil.wait_procs(alive, timeout=10)

def stream_training(command, log_path):
    with log_path.open("a", buffering=1) as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

print("Starting training; live output below. Console log:", output / "training-console.log", flush=True)
stream_training([
    sys.executable, "-u", script,
    "--bundle", "/content/layout_training",
    "--output", str(output),
], output / "training-console.log")


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

root = Path("/content/drive/MyDrive/MagiCanvas/audited_v2_candidates")
archive = Path("/content/audited_layout_adapters.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    import json
    specs = json.loads(Path("/content/layout_training/manifest.json").read_text())["models"]
    for spec in specs:
        if spec["key"] not in json.loads(Path("/content/layout_training/manifest.json").read_text())["training_queue"]:
            continue
        model = spec["key"]
        final = root / model / "final"
        assert (final / "adapter_model.safetensors").is_file(), model
        assert (final / "training_manifest.json").is_file(), model
        for p in sorted(final.iterdir()):
            if p.is_file():
                z.write(p, p.relative_to(root))
files.download(str(archive))
